<a href="https://colab.research.google.com/github/darshanananth/Symbol_detection/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: CLEAN Environment Setup

# 1. Wipe previous broken folders
%cd /content
!rm -rf GroundingDINO weights

# 2. Install packages directly via pip (bypasses the local C++ build crash)
!pip install -q git+https://github.com/IDEA-Research/GroundingDINO.git
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q git+https://github.com/openai/CLIP.git

# 3. Force upgrade supervision (ignore the red pip warning about 0.6.0, we NEED 0.19.0)
!pip install -q supervision==0.19.0

/content
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for groundingdino
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (groundingdino)
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [2]:
# Cell 2: Download Weights & Config
!mkdir -p /content/weights
%cd /content/weights

# Download Models
!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

# Download Config
!wget -q https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py

%cd /content/

/content/weights
/content


In [3]:
# Cell 3: Pipeline Implementation
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import clip

# Standardized Grounding DINO imports
from groundingdino.util.inference import load_model, predict as dino_predict
import groundingdino.datasets.transforms as T

# SAM imports
from segment_anything import sam_model_registry, SamPredictor

class PIDDigitizationPipeline:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading models to {self.device}...")

        # 1. Load Grounding DINO (Updated paths)
        dino_config = "/content/weights/GroundingDINO_SwinT_OGC.py"
        dino_weights = "/content/weights/groundingdino_swint_ogc.pth"
        self.dino = load_model(dino_config, dino_weights)

        # 2. Load SAM
        sam_checkpoint = "/content/weights/sam_vit_h_4b8939.pth"
        sam = sam_model_registry["vit_h"](checkpoint=sam_checkpoint).to(self.device)
        self.sam_predictor = SamPredictor(sam)

        # 3. Load CLIP
        self.clip_model, self.clip_preprocess = clip.load("ViT-B/32", device=self.device)
        print("Pipeline Ready! ✅")

    def load_image_for_dino(self, image_path):
        transform = T.Compose([
            T.RandomResize([800], max_size=1333),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        image_source = Image.open(image_path).convert("RGB")
        image = np.asarray(image_source)
        image_transformed, _ = transform(image_source, None)
        return image, image_transformed

    def get_grounding_boxes(self, image_transformed, prompts, box_threshold=0.3, text_threshold=0.25):
        boxes, logits, phrases = dino_predict(
            model=self.dino,
            image=image_transformed,
            caption=prompts,
            box_threshold=box_threshold,
            text_threshold=text_threshold
        )
        return boxes, phrases

    def get_sam_masks(self, image, dino_boxes):
        self.sam_predictor.set_image(image)
        h, w, _ = image.shape
        dino_boxes = dino_boxes * torch.Tensor([w, h, w, h])
        dino_boxes[:, :2] -= dino_boxes[:, 2:] / 2
        dino_boxes[:, 2:] += dino_boxes[:, :2]

        dino_boxes_device = torch.tensor(dino_boxes, device=self.sam_predictor.device)
        transformed_boxes = self.sam_predictor.transform.apply_boxes_torch(dino_boxes_device, image.shape[:2])

        masks, scores, _ = self.sam_predictor.predict_torch(
            point_coords=None,
            point_labels=None,
            boxes=transformed_boxes,
            multimask_output=False
        )
        return masks

    def validate_with_clip(self, image_bgr, mask, target_label):
        mask_np = mask[0].cpu().numpy()
        isolated_crop = np.zeros_like(image_bgr)
        isolated_crop[mask_np] = image_bgr[mask_np]

        pil_image = Image.fromarray(cv2.cvtColor(isolated_crop, cv2.COLOR_BGR2RGB))
        image_input = self.clip_preprocess(pil_image).unsqueeze(0).to(self.device)
        text_input = clip.tokenize([target_label, "clutter", "background lines"]).to(self.device)

        with torch.no_grad():
            image_features = self.clip_model.encode_image(image_input)
            text_features = self.clip_model.encode_text(text_input)
            similarity = (image_features @ text_features.T).softmax(dim=-1)

        return similarity[0][0].item() > 0.5

    def visualize_results(self, image, masks, boxes, phrases):
        plt.figure(figsize=(15, 10))
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        for mask in masks:
            mask_np = mask[0].cpu().numpy()
            color = np.random.random(3)
            plt.imshow(np.dstack((color[0]*mask_np, color[1]*mask_np, color[2]*mask_np, mask_np*0.5)))
        plt.axis('off')
        plt.title("Isolated P&ID Symbols")
        plt.show()

pipeline = PIDDigitizationPipeline()

Loading models to cuda...
final text_encoder_type: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AttributeError: 'BertModel' object has no attribute 'get_head_mask'

In [3]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Grounding DINO
from groundingdino.util.inference import load_model, load_image, predict, annotate

# SAM
from segment_anything import sam_model_registry, SamPredictor

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [9]:
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Clone GroundingDINO repository if it doesn't exist
if not os.path.exists("GroundingDINO"):
    !git clone https://github.com/IDEA-Research/GroundingDINO.git

# ---------- Grounding DINO ----------
DINO_CONFIG = "GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
DINO_WEIGHTS = "models/groundingdino_swint_ogc.pth"

dino_model = load_model(DINO_CONFIG, DINO_WEIGHTS)
dino_model = dino_model.to(DEVICE)

# ---------- SAM ----------
sam_checkpoint = "models/sam_vit_h_4b8939.pth"
sam = sam_model_registry["vit_h"](checkpoint=sam_checkpoint)
sam.to(device=DEVICE)
sam_predictor = SamPredictor(sam)

print("✅ All models loaded successfully")

Cloning into 'GroundingDINO'...
remote: Enumerating objects: 463, done.
remote: Total 463 (delta 0), reused 0 (delta 0), pack-reused 463 (from 1)
Receiving objects: 100% (463/463), 12.91 MiB | 16.34 MiB/s, done.
Resolving deltas: 100% (221/221), done.


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


final text_encoder_type: bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AttributeError: 'BertModel' object has no attribute 'get_head_mask'

In [4]:
# Upload your P&ID image
from google.colab import files
uploaded = files.upload()

IMAGE_PATH = list(uploaded.keys())[0]

# Load image
image_source, image = load_image(IMAGE_PATH)

# For SAM
image_bgr = cv2.imread(IMAGE_PATH)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

plt.imshow(image_rgb)
plt.axis("off")
plt.title("Input P&ID Image")

KeyboardInterrupt: 